In [1]:
from figgie_gym.pipelines.tensordict_dataloader import TensorDictDataModule


data_module = TensorDictDataModule(
    data_dir="/Users/yawerijaz/figgie_gym/data/multi_dev/tensordict",
    data_split_pct=(0.8, 0.1, 0.1),
    batch_size=1024,
    apply_soft_clip_on_prices=True,
    seed=42,
    preprocessor="nested",
)
data_module.setup()
td = data_module.datasets["train"]["x"]

2026-01-22 15:09:34 [info     ] Loading tensordict             datadir=/Users/yawerijaz/figgie_gym/data/multi_dev/tensordict
2026-01-22 15:09:48 [info     ] Preprocessing tensordict      
2026-01-22 15:09:48 [info     ] Splitting tensordict           total_experiments=100 train_size=80 val_size=10


In [41]:
from torch import stack
from jaxtyping import Float

trade_summaries = stack(list(td['per_suit', 'trade_summaries'].values())[:3], dim=-1).float()
private_suit_info = stack(list(td['per_suit'].exclude('trade_summaries').values())[:3], dim=-1).float()

In [6]:
import torch
from torch import Tensor, nn

class TradeSummaryEmbedding(nn.Module):
    "Embeds trade summaries using linear layers."
    def __init__(self, trade_summary_input_dim: int, trade_summary_embed_dim:int):
        super().__init__() # pyright: ignore[reportUnknownMemberType]
        self.model = nn.Linear(trade_summary_input_dim, trade_summary_embed_dim, bias=False)
        with torch.no_grad():
            self.model.weight.copy_(torch.eye(trade_summary_input_dim))
    def forward(self, x: Float[Tensor, 'batch suit player ts_input']) -> Float[Tensor, 'batch suit player ts_embed']:
        return self.model(x)

trade_summary_embedding = TradeSummaryEmbedding(3, 3)(trade_summaries)
trade_summary_embedding.shape # batch, suit, player, embed

torch.Size([16400, 4, 5, 3])

In [ ]:
# class SameSuitContextAttn(nn.Module):
#     """Contextualizes each player's suit holdings info with other players' holding in the same suit.

#     Think of it like adding all users' publicly known holdings in a same suit together.
#     Since the game is symmetric on players' seat, this layer is permutation invariant.v(?)

#     Equivariant layer within color - swapping Club and Spade is the same game with everything swapped at each decision point.
#     Invariant to opposite colored suits.

#     Equivariant with respect to colors. Swapping red and black changes the game symmetrically.

#     E.g. When contextualizing Clubs, swapping Diamonds and Hearts doesn't change the game.
#     But Swapping Clubs and Spades would change the game symmetrically.

#     """
#     def __init__(self, trade_summary_embed_dim: int, num_head: int) -> None:
#         super().__init__() # pyright: ignore[reportUnknownMemberType]
#         self.attention = nn.MultiheadAttention(trade_summary_embed_dim, num_head, batch_first=True,)

#     def forward(self, trade_summary_embedding: Float[Tensor, 'batch suit player ts_embed']) -> Float[Tensor, 'batch suit player ts_embed']:
#         batch_dim, suit_dim, player_dim, embed_dim = trade_summary_embedding.shape
#         same_suit_all_player = (
#         trade_summary_embedding
#                             .flatten(0,1) # ... x player x ts_embed
#                                 )
#         same_suit_all_player, _ = self.attention(same_suit_all_player, same_suit_all_player, same_suit_all_player)
#         all_suit_all_player = same_suit_all_player.reshape(batch_dim, suit_dim, player_dim, embed_dim)
#         trade_summary_embedding_enriched_with_all_holdings_in_same_suit = trade_summary_embedding + all_suit_all_player
#         return trade_summary_embedding_enriched_with_all_holdings_in_same_suit
    
class SameSuitContext(nn.Module):
    """Contextualizes each player's knowledge about a suit with other players' actions in the same suit.

    Think of it like adding all users' publicly known trades in a same suit together.
    Since the game is symmetric on players' seat, this layer is permutation invariant.
    """
    def __init__(self, trade_summary_embed_dim: int, known_trade_aggregate_embed_dim:int):
        super().__init__() # pyright: ignore[reportUnknownMemberType]
        self.prepool_model = nn.Linear(trade_summary_embed_dim, known_trade_aggregate_embed_dim, bias=False)
        with torch.no_grad():
            self.prepool_model.weight.copy_(torch.eye(trade_summary_embed_dim))
        
    def forward(self, trade_summary_embedding: Float[Tensor, 'batch suit player ts_embed']) -> Float[Tensor, 'batch suit known_trade_agg_embed']:
        prepool_transformed = self.prepool_model(trade_summary_embedding)
        return prepool_transformed.sum(dim=2)
    
known_trade_embed = SameSuitContext(3, 3)(trade_summary_embedding)
known_trade_embed.shape

torch.Size([16400, 4, 3])

In [66]:
class AllSingleSuitContext(nn.Module):
    """Combines player's proprietary holding information with pooled knowledge learned from other player's action."""
    def __init__(self, private_holding_in_dim: int, known_trade_aggregate_embed_dim: int, suit_embed_dim: int):
        super().__init__() # pyright: ignore[reportUnknownMemberType]
        self.model = nn.Linear(private_holding_in_dim + known_trade_aggregate_embed_dim, suit_embed_dim, bias=False)
        with torch.no_grad():
            self.model.weight.copy_(torch.ones((private_holding_in_dim + known_trade_aggregate_embed_dim, suit_embed_dim)).T)
        
    def forward(self, private_suit_info: Float[Tensor, 'batch suit private'], known_trade_embed: Float[Tensor, 'batch suit known_trade_agg_embed']) -> Float[Tensor, 'batch suit suit_embed_dim']:
        all_suit_info = torch.concat([private_suit_info, known_trade_embed], dim=-1)
        return self.model(all_suit_info)

suit_all_context = AllSingleSuitContext(3, 3, 7)(private_suit_info, known_trade_embed)


In [ ]:
class AllOtherSuitsContext(nn.Module):
    """Contextualize information regarding the other suit of the same colored.
    
    Equivariant layer within color - swapping Club and Spade is the same game with everything swapped at each decision point.

    Equivariant layer within color - swapping Club and Spade is the same game with everything swapped at each decision point.
    Invariant to opposite colored suits.

    Equivariant with respect to colors. Swapping red and black changes the game symmetrically.

    E.g. When contextualizing Clubs, swapping Diamonds and Hearts doesn't change the game.
    But Swapping Clubs and Spades would change the game symmetrically.

    Swapping Heart and Spade will be a totally different, unrelated game.

    A simple flat model for now.

    A simple way to do this is to group by color, and for each suit, concatenate the other suit of the same color, and the sum of all suits of the other color as context.

    """
    def __init__(self, suit_embed_dim: int):
        super().__init__() # pyright: ignore[reportUnknownMemberType]
        # self.same_color_model = nn.Linear(2*suit_embed_dim , suit_embed_dim)
        # self.opposite_color_model = nn.Linear(2*suit_embed_dim , suit_embed_dim)
        self.model = nn.Linear(3*suit_embed_dim , suit_embed_dim)

    def forward(self, suit_info: Float[Tensor, 'batch suit suit_embed_dim']) -> Float[Tensor, 'batch suit suit_embed_dim']:
        batch_dim, suit_dim, suit_embed_dim = suit_info.shape
        color_dim = 2 # red / black
        grouped_by_color = suit_info.reshape(batch_dim, color_dim, suit_dim // color_dim, suit_embed_dim)# batch color suit suit_embed_dim
        grouped_by_color_concat_with_other_suits = torch.concat([grouped_by_color, grouped_by_color.flip(2), grouped_by_color.flip(1).mean(dim=2, keepdim=True).repeat(1, 1, 2, 1)], dim=-1)
        grouped_by_color_with_all_suits_context = self.model(grouped_by_color_concat_with_other_suits)
        
        # same_color_suit = grouped_by_color.flip(2)
        # same_color_combined = torch.concat([grouped_by_color, same_color_suit], dim=-1)
        # grouped_by_color_with_same_color_context = self.same_color_model(same_color_combined)
        
        # opposite_color_suit_sum = grouped_by_color_with_same_color_context.sum(dim=2, keepdim=True).flip(1).repeat(1, 1, 2, 1)
        # opposite_color_combined = torch.concat([grouped_by_color_with_same_color_context, opposite_color_suit_sum], dim=-1)
        # grouped_by_color_with_all_suits_context = self.opposite_color_model(opposite_color_combined)
        
        suit_info_with_all_suit_context = grouped_by_color_with_all_suits_context.reshape(batch_dim, suit_dim, suit_embed_dim)
        return suit_info_with_all_suit_context
    
nnn = AllOtherSuitsContext(7, )
suit_all_context_ = torch.Tensor([0, 1, 2, 3]).repeat((2, 7, 1)).transpose(1, 2)
nnn(suit_all_context_)

tensor([[[-0.5431,  0.0109,  0.2704,  0.9788, -0.7921,  1.0687, -1.2643],
         [-1.1676, -0.1291,  0.5622,  1.5380, -0.9599,  0.3660, -0.9472],
         [-0.4109,  0.7174, -0.9259, -1.1303, -2.6667, -0.1630,  0.9871],
         [-1.0354,  0.5774, -0.6342, -0.5711, -2.8344, -0.8656,  1.3042]],

        [[-0.5431,  0.0109,  0.2704,  0.9788, -0.7921,  1.0687, -1.2643],
         [-1.1676, -0.1291,  0.5622,  1.5380, -0.9599,  0.3660, -0.9472],
         [-0.4109,  0.7174, -0.9259, -1.1303, -2.6667, -0.1630,  0.9871],
         [-1.0354,  0.5774, -0.6342, -0.5711, -2.8344, -0.8656,  1.3042]]],
       grad_fn=<ViewBackward0>)

In [235]:
suit_all_context_ = torch.Tensor([2, 3, 0.5, 0.5]).repeat((2, 7, 1)).transpose(1, 2)
nnn(suit_all_context_)

tensor([[[-0.4109,  0.7174, -0.9259, -1.1303, -2.6667, -0.1630,  0.9871],
         [-1.0354,  0.5774, -0.6342, -0.5711, -2.8344, -0.8656,  1.3042],
         [-0.8554, -0.0591,  0.4163,  1.2584, -0.8760,  0.7173, -1.1057],
         [-0.8554, -0.0591,  0.4163,  1.2584, -0.8760,  0.7173, -1.1057]],

        [[-0.4109,  0.7174, -0.9259, -1.1303, -2.6667, -0.1630,  0.9871],
         [-1.0354,  0.5774, -0.6342, -0.5711, -2.8344, -0.8656,  1.3042],
         [-0.8554, -0.0591,  0.4163,  1.2584, -0.8760,  0.7173, -1.1057],
         [-0.8554, -0.0591,  0.4163,  1.2584, -0.8760,  0.7173, -1.1057]]],
       grad_fn=<ViewBackward0>)

In [ ]:
# class OppositeColoredContext(nn.Module):
#     """Contextualize information regarding the suits of the opposite color.
    
#     Equivariant layer within color - swapping Club and Spade is the same game with everything swapped at each decision point.
#     Invariant to opposite colored suits.

#     Equivariant with respect to colors. Swapping red and black changes the game symmetrically.

#     E.g. When contextualizing Clubs, swapping Diamonds and Hearts doesn't change the game.
#     But Swapping Clubs and Spades would change the game symmetrically.
#     """
#     def __init__(self, suit_embed_dim: int):
#         super().__init__() # pyright: ignore[reportUnknownMemberType]
#         self.model = nn.Linear(2*suit_embed_dim , suit_embed_dim)

#     def forward(self, suit_info: Float[Tensor, 'batch suit suit_embed_dim']) -> Float[Tensor, 'batch suit suit_embed_dim']:
#         batch_dim, suit_dim, suit_embed_dim = suit_info.shape
#         color_dim = 2 # red / black
#         grouped_by_color = suit_info.reshape(batch_dim, color_dim, suit_dim // color_dim, suit_embed_dim)# batch color suit suit_embed_dim
#         the_other_suit_of_same_color = grouped_by_color.flip(2)
#         combined = torch.concat([grouped_by_color, the_other_suit_of_same_color], dim=-1)
#         grouped_by_color_with_context = self.model(combined)
#         # .sum(dim=2, keepdim=True).flip(1).repeat(1, 1, 2, 1)
#         suit_info_with_context = grouped_by_color_with_context.reshape(batch_dim, suit_dim, suit_embed_dim)
#         return suit_info_with_context
    

In [174]:
suit_all_context_2 = torch.Tensor([1, 0, 3, 2]).repeat((2, 7, 1)).transpose(1, 2)
nnn(suit_all_context_2)
# suit_all_context_2

tensor([[[ 0.6952, -0.3936, -0.1315, -0.1775, -0.1223,  0.1064, -0.3824],
         [ 0.1258,  0.1617,  1.1036,  0.1019, -0.9080,  0.0319, -0.4313],
         [ 1.4474, -1.3947,  1.6525, -0.6952, -1.3243, -0.5635, -2.6879],
         [ 0.8781, -0.8393,  2.8876, -0.4158, -2.1100, -0.6380, -2.7368]],

        [[ 0.6952, -0.3936, -0.1315, -0.1775, -0.1223,  0.1064, -0.3824],
         [ 0.1258,  0.1617,  1.1036,  0.1019, -0.9080,  0.0319, -0.4313],
         [ 1.4474, -1.3947,  1.6525, -0.6952, -1.3243, -0.5635, -2.6879],
         [ 0.8781, -0.8393,  2.8876, -0.4158, -2.1100, -0.6380, -2.7368]]],
       grad_fn=<ViewBackward0>)

In [134]:
torch.concat([like_color_grouped, like_color_grouped.flip(2)], dim=-1).shape

torch.Size([2, 2, 2, 8])

In [145]:
suit_all_context_ = torch.Tensor([0, 1, 2, 3]).repeat((2, 7, 1)).transpose(1, 2)
suit_all_context_

tensor([[[0., 0., 0., 0., 0., 0., 0.],
         [1., 1., 1., 1., 1., 1., 1.],
         [2., 2., 2., 2., 2., 2., 2.],
         [3., 3., 3., 3., 3., 3., 3.]],

        [[0., 0., 0., 0., 0., 0., 0.],
         [1., 1., 1., 1., 1., 1., 1.],
         [2., 2., 2., 2., 2., 2., 2.],
         [3., 3., 3., 3., 3., 3., 3.]]])

In [ ]:
suit_all_context[-2:, ...]
b_s_e = torch.Tensor([0, 1, 2, 3]).repeat((2, 4, 1)).transpose(1, 2)
print(b_s_e.shape)
like_color_grouped = b_s_e.reshape(2, 2, 2, 4)
# like_color_grouped
print(like_color_grouped.shape)
opposite_color_grouped_summary = like_color_grouped.flip(1).mean(dim=2, keepdim=True).repeat(1, 1, 2, 1)
opposite_color_grouped_summary
torch.concat([like_color_grouped, like_color_grouped.flip(2), like_color_grouped.flip(1).mean(dim=2, keepdim=True).repeat(1, 1, 2, 1)], dim=-1)
# print(like_color_grouped_summary.shape)
# like_color_grouped_with_context= like_color_grouped + like_color_grouped_summary
# like_color_grouped_with_context.reshape(2, 4, 4)


torch.Size([2, 4, 4])
torch.Size([2, 2, 2, 4])


torch.Size([2, 2, 2, 12])

In [132]:
like_color_grouped

tensor([[[[0., 0., 0., 0.],
          [1., 1., 1., 1.]],

         [[2., 2., 2., 2.],
          [3., 3., 3., 3.]]],


        [[[0., 0., 0., 0.],
          [1., 1., 1., 1.]],

         [[2., 2., 2., 2.],
          [3., 3., 3., 3.]]]])

In [ ]:
suit_all_context[-2:, ...]
b_s_e = torch.Tensor([0, 1, 2, 3]).repeat((2, 4, 1)).transpose(1, 2)
print(b_s_e.shape)
like_color_grouped = b_s_e.reshape(2, 2, 2, 4)
print(like_color_grouped.shape)
like_color_grouped_summary = (like_color_grouped + like_color_grouped.flip(2))/2
print(like_color_grouped_summary.shape)
like_color_grouped_with_context= like_color_grouped + like_color_grouped_summary
like_color_grouped_with_context.reshape(2, 4, 4)



torch.Size([2, 4, 4])
torch.Size([2, 2, 2, 4])
torch.Size([2, 2, 2, 4])


tensor([[[[0.5000, 0.5000, 0.5000, 0.5000],
          [1.5000, 1.5000, 1.5000, 1.5000]],

         [[4.5000, 4.5000, 4.5000, 4.5000],
          [5.5000, 5.5000, 5.5000, 5.5000]]],


        [[[0.5000, 0.5000, 0.5000, 0.5000],
          [1.5000, 1.5000, 1.5000, 1.5000]],

         [[4.5000, 4.5000, 4.5000, 4.5000],
          [5.5000, 5.5000, 5.5000, 5.5000]]]])

In [194]:
like_color_grouped_with_context.sum(dim=2, keepdim=True).flip(1).repeat(1, 1, 2, 1)

tensor([[[[10., 10., 10., 10.],
          [10., 10., 10., 10.]],

         [[ 2.,  2.,  2.,  2.],
          [ 2.,  2.,  2.,  2.]]],


        [[[10., 10., 10., 10.],
          [10., 10., 10., 10.]],

         [[ 2.,  2.,  2.,  2.],
          [ 2.,  2.,  2.,  2.]]]])

In [ ]:
# batch_dim, suit_dim, player_dim, embed_dim = trade_summary_embedding.shape

# single_suit_all_player_embed = (
#     trade_summary_embedding
#                             .flatten(0,1) # ... player embed
#                                 )
# single_suit_all_player_attn_out ,_=  nn.MultiheadAttention(3, 1, batch_first=True)(single_suit_all_player_embed,single_suit_all_player_embed,single_suit_all_player_embed)
# suit_info_with_info_from_all_players = single_suit_all_player_attn_out.reshape(batch_dim, suit_dim, player_dim, embed_dim)

# trade_summary_where_every_suit_has_info_across_players = trade_summary_embedding + suit_info_with_info_from_all_players

In [ ]:
batch_dim, suit_dim, player_dim, embed_dim = trade_summary_where_every_suit_has_info_across_players.shape

single_player_holdings = (
    trade_summary_where_every_suit_has_info_across_players # batch, suit, player, embed
    .permute(0, 2, 1, 3)  # batch, player, suit, embed
    .flatten(0, 1)  # batch * player, suit, embed
)

# full attention - no distingushing color
single_player_holdings_attn_out, _ = (
    nn.MultiheadAttention(3, 1, batch_first=True)(single_player_holdings,single_player_holdings,single_player_holdings))
player_holding_with_portfolio_context = (single_player_holdings_attn_out
    .reshape(batch_dim, player_dim, suit_dim, embed_dim)
    .permute(0, 2, 1, 3)
)
# player_holding_with_portfolio_context = trade_summary_embedding + player_holding_with_portfolio_context
# player_holding_with_portfolio_context.shape

In [46]:
?torch.permute

Docstring:
permute(input, dims) -> Tensor

Returns a view of the original tensor :attr:`input` with its dimensions permuted.

Args:
    input (Tensor): the input tensor.
    dims (tuple of int): The desired ordering of dimensions

Example:
    >>> x = torch.randn(2, 3, 5)
    >>> x.size()
    torch.Size([2, 3, 5])
    >>> torch.permute(x, (2, 0, 1)).size()
    torch.Size([5, 2, 3])
Type:      builtin_function_or_method